# Smart City Autonomous Driving - Max-Steer Arc Sweep Escape Implementation

Phiên bản sửa triệt để lỗi xe bị trôi/nhích dần cán qua vật cản:
1. **Lùi Xung Ga Khỏe (Kick-Start Reverse -0.38)**: Bắn ga lùi ban đầu -0.38 trong 0.25s để thắng ma sát tĩnh và mở khóa ESC lùi 100%, sau đó duy trì lùi -0.25 trong 1.3s (lùi lùi xa hẳn 30-40cm).
2. **Tiến Bẻ Kịch Lái Né Vắt Góc (Max-Steer Arc Sweep 1.5s với Steer ±1.0 & Throttle 0.22)**: Tiến với góc lái kịch tối đa (±1.0) và ga 0.22 trong 1.5s tạo thành đường cong rẽ né vòng qua hẳn vật cản.
3. **Phân tích ROI Nửa Trái / Nửa Phải (Left/Right ROI Scanning)**: Tự động xác định hướng thông thoáng để bẻ lái né chính xác.
4. **Khôi phục Ngưỡng phát hiện khoảng cách ban đầu (`min_bbox_area = 900`)**: Đảm bảo phát hiện biển báo từ xa và chuẩn xác.
5. **Thực thi Chỉ thị Rẽ định thời (Timed Turn Maneuver 1.1s)**: Duy trì bẻ ngoặt góc lái 1.1s liên tục.
6. **Khóa Cooldown 3.0s**: Ép xe duy trì chạy thẳng bám làn 3.0s sau khi rẽ.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
import cv2
import numpy as np
import onnxruntime as ort
import rospy
from sensor_msgs.msg import Image
from jetracer.nvidia_racecar import NvidiaRacecar

print(f"[+] ONNX Runtime Version: {ort.__version__}")

available_providers = ort.get_available_providers()
print(f"[+] Available Providers: {available_providers}")

if 'CUDAExecutionProvider' in available_providers or 'TensorrtExecutionProvider' in available_providers:
    print("[✓] Đã sẵn sàng chạy ONNX trên GPU TensorRT/CUDA!")
else:
    print("[!] Cảnh báo: Chạy ONNX trên CPU.")

In [ ]:
import gc
import time
import cv2
import numpy as np
import rospy
from IPython.display import Image, clear_output, display

from smart_city.camera_steam import CameraStream 
from smart_city.processor import YOLOProcessor, RoadProcessor
from smart_city.traffic_fsm import TrafficFSM
from smart_city.onnx_engine import ONNXEngine
from smart_city.controller import RacecarController

# 1. Khởi tạo Camera ROS Stream
cam = CameraStream(topic_name='/csi_cam_0/image_raw', width=500, height=300, record_video=False)

# 2. Khởi tạo Engine ONNX
traffic_engine = ONNXEngine('../models/best.onnx')
road_engine = ONNXEngine('../models/best_model_mobilenet.onnx')

# 3. Khởi tạo Processors & FSM với Ngưỡng ban đầu (min_bbox_area = 900)
traffic_processor = YOLOProcessor(img_size=640, conf_thresh=0.45)
road_processor = RoadProcessor(img_size=(224, 224), threshold=0.5)

fsm = TrafficFSM(
    default_state='FORWARD',
    conf_threshold=0.5,
    min_consecutive_frames=3,
    state_timeout=2.0,
    min_bbox_area=900,         # Ngưỡng diện tích ban đầu 900px²
    roi_x_min=0.05,            # ROI ngang ban đầu 0.05 - 0.95
    roi_x_max=0.95,
    cooldown_duration=3.0
)

# 4. Khởi tạo RacecarController với max_throttle 0.45
car = RacecarController(base_throttle=0.18, max_throttle=0.45)

print("✅ Khởi tạo Hệ thống Smart City Sửa Lỗi Né Vật Cản Cường Độ Cao thành công!")

In [ ]:
DISPLAY_UI, SKIP_FRAMES, TARGET_FPS = True, 1, 20

# ----------------------------------------------------
# CẤU HÌNH NÉ VẬT CẢN CƯỜNG ĐỘ CAO (MAX-STEER ARC SWEEP)
# ----------------------------------------------------
CONFIRM_FRAMES = 3
CONFIRM_THRESHOLD = 0.5

maneuver_state = 'DRIVE'       # States: 'DRIVE', 'ACTION_TURN_LEFT', 'ACTION_TURN_RIGHT', 'REVERSE_TURNING', 'PAUSE', 'FORWARD_ESCAPE'
maneuver_start_time = time.time()
blocked_frame_count = 0

reverse_steer = -1.0
forward_escape_steer = 1.0
best_open_side = 'LEFT'

display_handle = display(None, display_id=True) if DISPLAY_UI else None
rate = rospy.Rate(TARGET_FPS)
frame_count = 0
LOG_PANEL_WIDTH = 350

actual_steering = 0.0
actual_throttle = 0.0

rospy.loginfo("🚀 Bắt đầu Chạy Tự Hành Smart City (Max-Steer Arc Sweep Escape)...")

try:
    while not rospy.is_shutdown():
        frame = cam.get_frame()
        if frame is None:
            rospy.logwarn_throttle(2.0, "⏳ Đang chờ khung hình từ ROS Topic...")
            rate.sleep()
            continue

        start_time = rospy.get_time()
        frame_count += 1
        now = time.time()

        # --- 1. AI INFERENCE (YOLO & MOBILENET) ---
        t_input, orig_h, orig_w = traffic_processor.preprocess(frame)
        detections = traffic_processor.postprocess(traffic_engine.infer(t_input), orig_h, orig_w)

        r_input = road_processor.preprocess(frame)
        road_result = road_processor.postprocess(road_engine.infer(r_input))
        road_status, blocked_prob = road_result['status'], road_result['blocked_probability']

        # --- 2. DEBOUNCE BLOCK CONFIRMATION ---
        if blocked_prob >= CONFIRM_THRESHOLD:
            blocked_frame_count += 1
        else:
            blocked_frame_count = max(0, blocked_frame_count - 1)

        # --- 3. MÁY TRẠNG THÁI ĐIỀU KHIỂN CHÍNH (MAIN CONTROL STATE MACHINE) ---
        
        # A. TRẠNG THÁI LÁI BÌNH THƯỜNG (DRIVE)
        if maneuver_state == 'DRIVE':
            # ƯU TIÊN 1: Nếu dính BLOCKED liên tục 3 frame -> Kích hoạt Né vật cản Cường Độ Cao (Max-Steer Arc Sweep)
            if blocked_frame_count >= CONFIRM_FRAMES:
                # Cắt ROI Left/Right để xác định phía nào thông thoáng hơn hẳn
                side_roi = road_processor.process_left_right_roi(frame, road_engine)
                best_open_side = side_roi['best_side']

                if best_open_side == 'LEFT':
                    # Phía TRÁI trống hơn -> Lùi bẻ kịch phải (+1.0), Tiến bẻ kịch trái (-1.0)
                    reverse_steer = 1.0
                    forward_escape_steer = -1.0
                else:
                    # Phía PHẢI trống hơn -> Lùi bẻ kịch trái (-1.0), Tiến bẻ kịch phải (+1.0)
                    reverse_steer = -1.0
                    forward_escape_steer = 1.0

                maneuver_state = 'REVERSE_TURNING'
                maneuver_start_time = now
                actual_steering = reverse_steer
                actual_throttle = -0.38  # Kick reverse pulse
                car.set_steering(actual_steering)
                car.set_throttle(actual_throttle)
            else:
                # Cập nhật trạng thái FSM biển báo
                fsm_action = fsm.update(detections, img_w=orig_w, img_h=orig_h)

                # ƯU TIÊN 2: Xử lý chỉ thị biển báo giao thông
                if fsm_action == 'TURN_LEFT':
                    side_roi = road_processor.process_left_right_roi(frame, road_engine)
                    if side_roi['prob_left'] < 0.5: # Phía bên trái thực sự có đường!
                        maneuver_state = 'ACTION_TURN_LEFT'
                        maneuver_start_time = now
                        actual_steering = -0.85
                        actual_throttle = 0.20
                    else:
                        # BIỂN GIẢ (Bên trái là tường/lề) -> Hủy rẽ, bật Cooldown 3.0s và đi thẳng
                        fsm.trigger_cooldown()
                        actual_steering = 0.0
                        actual_throttle = 0.18
                    car.set_steering(actual_steering)
                    car.set_throttle(actual_throttle)

                elif fsm_action == 'TURN_RIGHT':
                    side_roi = road_processor.process_left_right_roi(frame, road_engine)
                    if side_roi['prob_right'] < 0.5: # Phía bên phải thực sự có đường!
                        maneuver_state = 'ACTION_TURN_RIGHT'
                        maneuver_start_time = now
                        actual_steering = 0.85
                        actual_throttle = 0.20
                    else:
                        # BIỂN GIẢ -> Hủy rẽ, bật Cooldown 3.0s và đi thẳng
                        fsm.trigger_cooldown()
                        actual_steering = 0.0
                        actual_throttle = 0.18
                    car.set_steering(actual_steering)
                    car.set_throttle(actual_throttle)

                elif fsm_action == 'PROHIBITION':
                    # Biển Cấm Đi Thẳng: Kiểm tra bên trái và phải bên nào trống hơn để rẽ
                    side_roi = road_processor.process_left_right_roi(frame, road_engine)
                    if side_roi['prob_left'] < 0.5:
                        maneuver_state = 'ACTION_TURN_LEFT'
                        maneuver_start_time = now
                        actual_steering = -0.85
                        actual_throttle = 0.20
                    elif side_roi['prob_right'] < 0.5:
                        maneuver_state = 'ACTION_TURN_RIGHT'
                        maneuver_start_time = now
                        actual_steering = 0.85
                        actual_throttle = 0.20
                    else:
                        car.stop()
                        actual_steering = 0.0
                        actual_throttle = 0.0

                elif fsm_action in ['STOP', 'RED_LIGHT']:
                    car.stop()
                    actual_steering = 0.0
                    actual_throttle = 0.0
                else:
                    # Đường trống bình thường -> Chạy tiến thẳng
                    actual_steering = 0.0
                    actual_throttle = 0.18
                    car.set_steering(actual_steering)
                    car.set_throttle(actual_throttle)

        # B. THỰC THI CHỈ THỊ RẼ TRÁI ĐỊNH THỜI (1.1s)
        elif maneuver_state == 'ACTION_TURN_LEFT':
            if now - maneuver_start_time < 1.1:
                actual_steering = -0.85
                actual_throttle = 0.20
                car.set_steering(actual_steering)
                car.set_throttle(actual_throttle)
            else:
                fsm.trigger_cooldown()
                maneuver_state = 'DRIVE'
                actual_steering = 0.0
                actual_throttle = 0.18
                car.set_steering(actual_steering)
                car.set_throttle(actual_throttle)

        # C. THỰC THI CHỈ THỊ RẼ PHẢI ĐỊNH THỜI (1.1s)
        elif maneuver_state == 'ACTION_TURN_RIGHT':
            if now - maneuver_start_time < 1.1:
                actual_steering = 0.85
                actual_throttle = 0.20
                car.set_steering(actual_steering)
                car.set_throttle(actual_throttle)
            else:
                fsm.trigger_cooldown()
                maneuver_state = 'DRIVE'
                actual_steering = 0.0
                actual_throttle = 0.18
                car.set_steering(actual_steering)
                car.set_throttle(actual_throttle)

        # D. LÙI CƯỜNG ĐỘ CAO NÉ VẬT CẢN (REVERSE_TURNING 1.3s: 0.25s Kick -0.38, 1.05s Settle -0.25)
        elif maneuver_state == 'REVERSE_TURNING':
            elapsed = now - maneuver_start_time
            if elapsed < 1.3:
                actual_steering = reverse_steer
                actual_throttle = -0.38 if elapsed < 0.25 else -0.25 # Kick reverse để lùi xa hẳn 30-40cm
            else:
                maneuver_state = 'PAUSE'
                maneuver_start_time = now
                actual_steering = 0.0
                actual_throttle = 0.0
            car.set_steering(actual_steering)
            car.set_throttle(actual_throttle)

        # E. DỪNG NGHỈ 0.2s
        elif maneuver_state == 'PAUSE':
            if now - maneuver_start_time < 0.2:
                actual_steering = 0.0
                actual_throttle = 0.0
            else:
                maneuver_state = 'FORWARD_ESCAPE'
                maneuver_start_time = now
                actual_steering = forward_escape_steer
                actual_throttle = 0.23
            car.set_steering(actual_steering)
            car.set_throttle(actual_throttle)

        # F. TIẾN BẺ KỊCH LÁI RẼ VÒNG NÉ VẬT CẢN (FORWARD_ESCAPE 1.5s với Steer ±1.0 & Throttle 0.23)
        elif maneuver_state == 'FORWARD_ESCAPE':
            if now - maneuver_start_time < 1.5:
                actual_steering = forward_escape_steer  # Kịch lái ±1.0
                actual_throttle = 0.23                  # Ga khỏe 0.23 quét đường cong né vòng
            else:
                maneuver_state = 'DRIVE'
                blocked_frame_count = 0
                actual_steering = 0.0
                actual_throttle = 0.18
            car.set_steering(actual_steering)
            car.set_throttle(actual_throttle)

        # --- 4. METRICS & LOGGING ---
        latency_ms = (rospy.get_time() - start_time) * 1000
        fps_real = 1000 / max(latency_ms, 1)

        if frame_count % 50 == 0:
            clear_output(wait=True)
            if DISPLAY_UI:
                display_handle = display(None, display_id=True)

        # --- 5. UI DASHBOARD DISPLAY ---
        if DISPLAY_UI and (frame_count % SKIP_FRAMES == 0):
            debug_frame = traffic_processor.draw_bboxes(frame, detections)

            h, w, c = debug_frame.shape
            log_panel = np.zeros((h, LOG_PANEL_WIDTH, c), dtype=np.uint8)

            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.5
            font_thickness = 1
            text_color = (255, 255, 255)
            status_color = (0, 0, 255) if maneuver_state != 'DRIVE' else (0, 255, 0)

            y_offset = 25
            line_height = 22

            cv2.putText(log_panel, "=== SMART CITY DASHBOARD ===", (10, y_offset), font, 0.55, (0, 255, 255), 2)
            y_offset += 35

            cv2.putText(log_panel, "FSM State: ", (10, y_offset), font, font_scale, text_color, font_thickness)
            cv2.putText(log_panel, f"{fsm.current_state}", (110, y_offset), font, font_scale, (255, 255, 0), 2)
            y_offset += line_height

            cv2.putText(log_panel, "Maneuver: ", (10, y_offset), font, font_scale, text_color, font_thickness)
            cv2.putText(log_panel, f"{maneuver_state}", (110, y_offset), font, font_scale, status_color, 2)
            y_offset += line_height

            cv2.putText(log_panel, f"Open Side: {best_open_side}", (10, y_offset), font, font_scale, (0, 255, 255), font_thickness)
            y_offset += line_height

            cv2.putText(log_panel, f"Road: {road_status} ({blocked_prob:.2f})", (10, y_offset), font, font_scale, text_color, font_thickness)
            y_offset += line_height

            cv2.putText(log_panel, "--- Controls ---", (10, y_offset), font, font_scale, (150, 150, 150), font_thickness)
            y_offset += line_height
            cv2.putText(log_panel, f"Steering: {actual_steering:+.2f}", (20, y_offset), font, font_scale, text_color, font_thickness)
            y_offset += line_height
            cv2.putText(log_panel, f"Throttle: {actual_throttle:+.2f}", (20, y_offset), font, font_scale, text_color, font_thickness)
            y_offset += 30

            cv2.putText(log_panel, "--- Performance ---", (10, y_offset), font, font_scale, (150, 150, 150), font_thickness)
            y_offset += line_height
            cv2.putText(log_panel, f"FPS: {fps_real:.1f} | Frame: {frame_count}", (10, y_offset), font, font_scale, text_color, font_thickness)
            y_offset += line_height
            cv2.putText(log_panel, f"Latency: {latency_ms:.1f} ms", (10, y_offset), font, font_scale, text_color, font_thickness)

            if len(detections) > 0:
                y_offset += 30
                cv2.putText(log_panel, "--- Detections ---", (10, y_offset), font, font_scale, (150, 150, 150), font_thickness)
                y_offset += line_height
                for d in detections[:4]:
                    det_str = f"- {d['class_name']} ({d['confidence']:.2f})"
                    cv2.putText(log_panel, det_str, (10, y_offset), font, 0.4, (0, 200, 0), font_thickness)
                    y_offset += 18

            final_display_frame = np.hstack((debug_frame, log_panel))

            rgb_frame = cv2.cvtColor(final_display_frame, cv2.COLOR_BGR2RGB)
            _, jpeg = cv2.imencode('.jpg', rgb_frame, [int(cv2.IMWRITE_JPEG_QUALITY), 70])
            display_handle.update(Image(data=jpeg.tobytes()))

        if frame_count % 100 == 0:
            gc.collect()

        rate.sleep()

except KeyboardInterrupt:
    car.stop()
    rospy.loginfo("🛑 Đã dừng xe do KeyboardInterrupt.")
finally:
    car.stop()
    rospy.loginfo("🛑 Kết thúc chương trình Smart City Improved.")

In [ ]:
car.stop()
print("[*] Robot Emergency Stopped.")